# Train Semi V2 — RETFound DR Grading

Pipeline này dùng trực tiếp `checkpoint-best.pth` của `ai/grading`, EMA teacher tạo pseudo-label online cho ảnh ngoài, student học weak/strong augmentation và lưu đầy đủ artifact lên Drive.

Chạy tuần tự **Cell 1 → 6 → Train or Resume**. Nếu Colab bị ngắt, cell train tự tiếp tục từ `last.pth`.

In [ ]:
# CELL 1 — GPU diagnostics
import torch, sys, platform
from datetime import datetime, timezone
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('=== RUNTIME DIAGNOSTICS ===')
print('UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/dr_semi_v2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

In [ ]:
# CELL 3 — Source code và dependencies
import os, subprocess
GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/semi-v3-fixmatch-ema'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
print('Source ready:', REPO_DIR)

In [ ]:
# CELL 4 — Tải 2 dataset Kaggle và cấu hình train/resume/test
import getpass
from google.colab import userdata

GRADE_CHECKPOINT = Path('/content/drive/MyDrive/retfound_merged_seed42/checkpoint-best.pth')
DATA_ROOT = Path('/content/datasets')
LABELED_DATASET_REF = 'sehastrajits/fundus-aptosddridirdeyepacsmessidor'
UNLABELED_DATASET_REF = 'gzuidhof/diabetic-retinopathy-detection-resized'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}

def count_images(root):
    return sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def find_image_directory(root, directory_name):
    candidates = [p for p in root.rglob(directory_name) if p.is_dir()]
    scored = []
    for candidate in candidates:
        direct_images = sum(1 for p in candidate.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        scored.append((direct_images, candidate))
    if not scored or max(scored, key=lambda item: item[0])[0] == 0:
        raise RuntimeError(f'Không tìm thấy thư mục {directory_name} chứa ảnh trực tiếp trong {root}')
    direct_images, selected = max(scored, key=lambda item: item[0])
    print(f'✅ Chọn {selected} với {direct_images:,} ảnh không nhãn')
    return selected

def ensure_kaggle_dataset(dataset_ref, target):
    target.mkdir(parents=True, exist_ok=True)
    existing_count = count_images(target)
    if existing_count:
        print(f'✅ Đã có {existing_count:,} ảnh: {target}')
        return target
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if not token:
        try:
            token = userdata.get('KAGGLE_API_TOKEN')
        except Exception:
            token = getpass.getpass('Dán Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Thiếu KAGGLE_API_TOKEN trong Colab Secrets')
    os.environ['KAGGLE_API_TOKEN'] = token
    kaggle_cli = [sys.executable, '-m', 'kaggle']
    print(f'⬇️ Đang tải Kaggle dataset {dataset_ref} ...')
    subprocess.run(kaggle_cli + ['datasets', 'download', '-d', dataset_ref, '-p', str(target), '--unzip'], check=True)
    image_count = count_images(target)
    if not image_count:
        raise RuntimeError(f'Tải xong nhưng không tìm thấy ảnh trong {target}')
    print(f'✅ Sẵn sàng {image_count:,} ảnh: {target}')
    return target

labeled_download = ensure_kaggle_dataset(LABELED_DATASET_REF, DATA_ROOT / 'fundus_merged')
LABELED_DATASET_DIR = labeled_download / 'split_dataset'
eyepacs_download = ensure_kaggle_dataset(UNLABELED_DATASET_REF, DATA_ROOT / 'eyepacs2015_resized')
UNLABELED_DIR = find_image_directory(eyepacs_download, 'test_images_512')
RUN_NAME = 'run-eyepacs2015-fixmatch'
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME
PSEUDO_CACHE_DIR = DRIVE_ROOT / 'cache'

SEMI_CONFIG = {
    'epochs': 8,
    'patience': 4,
    'batch_size': 4,
    'accum_steps': 4,
    'head_lr': 1e-5,
    'backbone_lr': 1e-6,
    'min_lr': 1e-7,
    'weight_decay': 0.05,
    'training_mode': 'fixmatch',
    # Ngưỡng lần lượt cho grade 0, 1, 2, 3, 4.
    # Grade 0 cao hơn để hạn chế ảnh bình thường lấn át các grade bệnh.
    'grade_thresholds': [0.99, 0.90, 0.95, 0.90, 0.93],
    'unlabeled_batch_size': 4,
    'unsupervised_weight': 0.5,
    'unsupervised_warmup_epochs': 3,
    'ema_decay': 0.999,
    'num_workers': 2,
    'seed': 42,
}

for label, path in [('grade checkpoint', GRADE_CHECKPOINT), ('labeled dataset', LABELED_DATASET_DIR), ('unlabeled pool', UNLABELED_DIR)]:
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy {label}: {path}')
print('Grade checkpoint:', GRADE_CHECKPOINT)
print('Labeled replay:', LABELED_DATASET_DIR)
print('Unlabeled pool:', UNLABELED_DIR)
print('Output:', OUTPUT_DIR)
print('Reusable pseudo cache:', PSEUDO_CACHE_DIR)
print('Config:', SEMI_CONFIG)

In [ ]:
# CELL 5 — Kiểm tra contract của checkpoint grade
import json
checkpoint_metadata = torch.load(GRADE_CHECKPOINT, map_location='cpu', weights_only=False)
grade_args = checkpoint_metadata.get('args', {})
required = ['model_source', 'image_size', 'loss']
missing = [key for key in required if key not in grade_args]
if missing:
    raise ValueError(f'Checkpoint grade thiếu metadata bắt buộc: {missing}')
inferred_preprocessing = grade_args.get('preprocessing')
if inferred_preprocessing is None:
    inferred_preprocessing = 'ben_graham' if grade_args.get('enhance', False) else 'rgb_crop'
    print(f'⚠️ Checkpoint cũ thiếu preprocessing; suy ra: {inferred_preprocessing}')
print(json.dumps({
    'epoch': checkpoint_metadata.get('epoch'),
    'best_qwk': checkpoint_metadata.get('best_qwk'),
    'model_source': grade_args.get('model_source'),
    'architecture': grade_args.get('architecture'),
    'model_name': grade_args.get('model_name'),
    'image_size': grade_args.get('image_size'),
    'preprocessing': inferred_preprocessing,
    'loss': grade_args.get('loss'),
    'grading_contract': checkpoint_metadata.get('grading_contract', {}),
}, indent=2, ensure_ascii=False, default=str))
del checkpoint_metadata

In [ ]:
# CELL 6 — Command builder và live logger
import shlex, signal

def build_command(*, resume=None, eval_only=False):
    c = SEMI_CONFIG
    command = [
        sys.executable, '-u', '-m', 'ai.train_semi_v2.train',
        '--checkpoint', str(GRADE_CHECKPOINT),
        '--dataset-dir', str(LABELED_DATASET_DIR),
        '--unlabeled-dir', str(UNLABELED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--pseudo-cache-dir', str(PSEUDO_CACHE_DIR),
        '--epochs', str(c['epochs']), '--patience', str(c['patience']),
        '--batch-size', str(c['batch_size']), '--accum-steps', str(c['accum_steps']),
        '--head-lr', str(c['head_lr']), '--backbone-lr', str(c['backbone_lr']),
        '--min-lr', str(c['min_lr']), '--weight-decay', str(c['weight_decay']),
        '--training-mode', c['training_mode'],
        '--grade-thresholds', ','.join(map(str, c['grade_thresholds'])),
        '--unlabeled-batch-size', str(c['unlabeled_batch_size']),
        '--unsupervised-weight', str(c['unsupervised_weight']),
        '--unsupervised-warmup-epochs', str(c['unsupervised_warmup_epochs']),
        '--ema-decay', str(c['ema_decay']),
        '--num-workers', str(c['num_workers']), '--seed', str(c['seed']),
    ]
    if resume is not None:
        command.extend(['--resume', str(resume)])
    if eval_only:
        command.append('--eval-only')
    return command

def run_streaming(command, log_name):
    is_fresh_run = '--resume' not in command
    log_path = DRIVE_ROOT / f'{RUN_NAME}-{log_name}' if is_fresh_run else OUTPUT_DIR / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('COMMAND:', shlex.join(command))
    print('LIVE LOG:', log_path)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log_path.open('a', encoding='utf-8') as handle:
            for line in process.stdout:
                timestamped = f'[{datetime.now(timezone.utc).isoformat()}] {line}'
                print(timestamped, end='', flush=True)
                handle.write(timestamped)
                handle.flush()
        process.wait()
    except KeyboardInterrupt:
        print('\nĐang dừng an toàn. progress.csv hoặc last.pth vẫn được giữ để chạy tiếp.')
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait()
    if process.returncode not in (0, 130, -2):
        raise RuntimeError(f'Process thất bại với exit code {process.returncode}')
    return process.returncode

## Train hoặc tiếp tục tự động

Dùng cùng một cell cho cả lần đầu và sau khi Colab bị ngắt. Cache hoàn chỉnh sẽ bỏ qua dự đoán; cache dở dang tiếp tục từ ảnh chưa xử lý; nếu có `last.pth` thì tiếp tục epoch kế tiếp.

In [ ]:
# TRAIN OR RESUME
LAST_CHECKPOINT = OUTPUT_DIR / 'last.pth'
if LAST_CHECKPOINT.is_file():
    print('Tiếp tục train từ:', LAST_CHECKPOINT)
    run_streaming(build_command(resume=LAST_CHECKPOINT), 'resume.log')
else:
    print('Train FixMatch mới với pseudo-label online từ EMA teacher.')
    run_streaming(build_command(), 'run.log')

## Resume sau khi ngắt

Chạy cell này thay cho Train new. Student, EMA teacher, optimizer, scheduler, scaler, epoch và patience đều được khôi phục.

In [ ]:
# RESUME SEMI V2
LAST_CHECKPOINT = OUTPUT_DIR / 'last.pth'
if not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint resume: {LAST_CHECKPOINT}')
run_streaming(build_command(resume=LAST_CHECKPOINT), 'notebook-resume-live.log')

## Đánh giá held-out test

Chỉ chạy thủ công sau khi đã chọn checkpoint. Test không tham gia train, early stopping hoặc chọn model.

In [ ]:
# TEST CHECKPOINT
TEST_CHECKPOINT_KIND = 'best'  # 'best' hoặc 'last'
TEST_CHECKPOINT = OUTPUT_DIR / f'{TEST_CHECKPOINT_KIND}.pth'
if not TEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Không tìm thấy: {TEST_CHECKPOINT}')
run_streaming(build_command(resume=TEST_CHECKPOINT, eval_only=True), 'notebook-test-live.log')
for artifact in ['test_metrics.json', 'test_predictions.csv', 'summary.json']:
    path = OUTPUT_DIR / artifact
    if path.is_file():
        print(f'\n=== {artifact} ===')
        print(path.read_text(encoding='utf-8')[:5000] if path.suffix != '.csv' else path)

## Theo dõi pseudo-label online

- FixMatch không tạo cache pseudo-label tĩnh: EMA teacher dự đoán lại ở từng batch.
- `history.jsonl` ghi acceptance rate và số pseudo-label được nhận theo từng grade.
- Đổi ngưỡng giữa một run đang resume bị chặn để bảo đảm thí nghiệm nhất quán; hãy dùng `RUN_NAME` mới.

In [ ]:
# FIXMATCH HISTORY — chỉ đọc
history_path = OUTPUT_DIR / 'history.jsonl'
if history_path.is_file():
    records = [json.loads(line) for line in history_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for record in records:
        print('epoch', record['epoch'] + 1, 'acceptance=', round(record['pseudo_acceptance_rate'], 4), 'per_grade=', record['accepted_pseudo_labels_per_grade'])
else:
    print('Chưa có history.jsonl')